# Course Recommendation System

This notebook contains the main logic for the course recommendation system using Flask.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import neattext.functions as nfx
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

In [ ]:
# Load the dataset
df = pd.read_csv('UdemyCleanedTitle.csv')
print(f"Dataset shape: {df.shape}")
print("\nColumns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

In [ ]:
# Clean the course titles
def getcleantitle(df):
    df['Clean_title'] = df['course_title'].apply(nfx.remove_stopwords)
    df['Clean_title'] = df['Clean_title'].apply(nfx.remove_special_characters)
    return df

df_clean = getcleantitle(df)
print("Sample of cleaned titles:")
print(df_clean[['course_title', 'Clean_title']].head())

In [ ]:
# Create cosine similarity matrix
def getcosinemat(df):
    countvect = CountVectorizer()
    cvmat = countvect.fit_transform(df['Clean_title'])
    return cvmat

def cosinesimmat(cv_mat):
    return cosine_similarity(cv_mat)

cvmat = getcosinemat(df_clean)
cosine_mat = cosinesimmat(cvmat)
print(f"Cosine similarity matrix shape: {cosine_mat.shape}")

In [ ]:
# Recommendation function
def recommend_course(df, title, cosine_mat, numrec=6):
    course_index = pd.Series(df.index, index=df['course_title']).drop_duplicates()
    
    if title not in course_index:
        return f"Course '{title}' not found in dataset"
    
    index = course_index[title]
    scores = list(enumerate(cosine_mat[index]))
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    selected_course_index = [i[0] for i in sorted_scores[1:numrec+1]]
    selected_course_score = [i[1] for i in sorted_scores[1:numrec+1]]
    
    rec_df = df.iloc[selected_course_index]
    rec_df['Similarity_Score'] = selected_course_score
    
    final_recommended_courses = rec_df[['course_title', 'Similarity_Score', 'url', 'price', 'num_subscribers']]
    return final_recommended_courses

# Test the recommendation
test_course = df['course_title'].iloc[0]
print(f"Testing recommendations for: {test_course}")
recommendations = recommend_course(df_clean, test_course, cosine_mat)
recommendations

In [ ]:
# Search functionality for partial matches
def searchterm(term, df):
    result_df = df[df['course_title'].str.contains(term, case=False, na=False)]
    top6 = result_df.sort_values(by='num_subscribers', ascending=False).head(6)
    return top6

# Test search
search_term = "Python"
search_results = searchterm(search_term, df_clean)
print(f"Search results for '{search_term}':")
search_results[['course_title', 'num_subscribers', 'price']]